In [ ]:
#TODO make sure this renders in the github repo

# SwiGLU Feed Forward

![SwiGLU_FFN](../showcase_images/SwiGLU_FFN.png)

**From Llama 1:** "SwiGLU activation function [PaLM]. We replace the ReLU non-linearity by the SwiGLU activation function, introduced by [Shazeer (2020)](https://arxiv.org/abs/2002.05202) to improve the performance. We use a dimension of $\frac{2}{3} 4d$ instead of $4d$ as in PaLM."

**From [Google PaLM paper](https://arxiv.org/abs/2204.02311):** "**No biases** were used in any of the dense kernels or layer norms. We found this to result in increased training stability for large models."
- Many LLMs adopted this no bias approach.

**From [SwiGLU paper]((https://arxiv.org/abs/2002.05202)):** 
- "In this paper, we propose additional variations on the Transformer FFN layer which use GLU or one of its variants in place of the first linear transformation and the activation function. Again, we omit the **bias** terms."
- "Like ReLU, Swish is unbounded above and bounded below. Unlike ReLU, Swish is smooth and nonmonotonic. In fact, the non-monotonicity property of Swish distinguishes itself from most common activation functions."
- "Our experiments show that Swish consistently outperforms or matches the ReLU function on a variety of deep models"

**SwiGLU:**

$$\text{SwiGLU}(x, W, V) = \text{Swish}_{1}(xW) \otimes (xV)$$

$$\text{Swish} = x * \sigma(x)$$

$$\sigma(x) = (1 + \text{exp}(-x))^{-1}$$

- $\sigma(x) =$ is the Sigmoid function.
- No bias term is used.

SwiGLU consists three linear transformations (Gate, Up, Down) that run in parallel:
1. SwiGLU is applied to each position separately and identically.
2. The input is expanded into two distinct linear projections: the **Gate** and the **Up** projections. Both expand the dimension from $d_{model}$ to $d_{ff}$.
   1. By expanding we give the model more parameters to learn non-linear functions. This is also where the heavy pattern recognition happens.
3. The Swish activation is only applied to the **Gate** projection.
4. The activated **Gate** is multiplied by element-wise to the **Gate** projection.
5. A third linear transformation, the **Down** projection, contracts the tensor from $d_{ff}$ back to $d_{model}$ dimensions.

In [6]:
import EasyJupyter
import torch
import torch.nn as nn
import torch.nn.functional as F

In [7]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_config import BaseConfig

In [8]:
class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg: BaseConfig):
        """
        SwiGLU Feed Forward Neural Network class.
        """
        super().__init__()

        # We expand the dimensions of the linear layers to d_ff
        self.gate_project = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up_project = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)

        # Project the expanded dimensions from d_ff back to the d_model
        self.down_project = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: The input tensor of shape (batch_size, context_len, d_model).
        Returns:
            The output tensor of shape (batch_size, context_len, d_model).
        """
        # Calculate the gate and apply SiLU (Swish with beta=1)
        gate = F.silu(self.gate_project(x))

        # Calculate the up projection (un-activated)
        up = self.up_project(x)

        # Element-wise multiplication fo gate and up projection, then project back to the original dimensions
        return self.down_project(gate * up)
        

In [9]:
# @i-c
# TEST
from llama_config import BaseConfig

cfg = BaseConfig()
cfg.d_ff = 1024
cfg.d_model = 512
batch_size = 1
context_len = 24

SwiGLU = SwiGLUFeedForward(cfg)
dummy_input = torch.randn(batch_size, context_len, cfg.d_model)
output = SwiGLU(dummy_input)
f"Output shape: {output.shape}"


Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM


'Output shape: torch.Size([1, 24, 512])'